# 06 - Tuning and model selection

Goal: improve validation results and choose the best model. Since no model beat the baseline in notebook 05, I test a log target and a different loss function. I also check three previous choices: training period, `sys_created_by` and resolved incidents. The test set is not used.

## 1. Setup

In [1]:
import sys
import time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # to import src from the notebooks folder

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline
from src import config, features as ft, metrics, pipelines as pp

pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 90)
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

TARGET, FEATURES, RS = config.TARGET, config.FEATURES, config.RANDOM_STATE
t = ft.load_table()
assert t.shape == (24905, 25)    # table of notebook 03, time features, split
train = t[t["split"] == "train"]
val = t[t["split"] == "validation"]   # the test part is not used in this notebook


def make_model(name, params):
    """Model with its settings. `log_target` trains on log(1 + hours) and converts back to hours."""
    params = dict(params)
    log_target = params.pop("log_target")
    if name == "Ridge":
        model = Ridge(**params)
    elif name == "Random Forest":
        model = RandomForestRegressor(n_jobs=-1, random_state=RS, **params)
    else:
        model = HistGradientBoostingRegressor(early_stopping=False, random_state=RS, **params)
    if log_target:
        model = TransformedTargetRegressor(model, func=np.log1p, inverse_func=np.expm1)
    return model


def fit_eval(model, features=FEATURES, train_df=train):
    """Fit the pipeline on the train data and score it on validation, in hours."""
    pipe = Pipeline([("prep", pp.make_preprocessor(features)), ("model", model)])
    start = time.time()
    pipe.fit(train_df[features], train_df[TARGET])
    seconds = time.time() - start
    pred = np.clip(pipe.predict(val[features]), 0, None)
    scores = metrics.regression_metrics(val[TARGET], pred)
    return {**scores, "mean error": (pred - val[TARGET]).mean(), "fit seconds": seconds}, pred

pandas 2.3.3 | scikit-learn 1.7.2


## 2. Random search

Random search is used to find the best parameters for each model. It also tests `log_target` and, for HistGradientBoosting, different loss functions.

Each configuration is trained on the training set and evaluated on the validation set. I use **MAE** to select the best model because it is easier to interpret in hours and is less affected by extreme values than RMSE. RMSE and RMSLE are also calculated. The mean error shows whether the model tends to overestimate or underestimate the predictions.

In [2]:
SPACES = {
    "Ridge": {"alpha": [0.1, 1, 10, 100], "log_target": [False, True]},
    "Random Forest": {"n_estimators": [100, 200], "max_depth": [10, 20, None], "min_samples_split": [2, 10, 30],
                      "min_samples_leaf": [1, 5, 20], "log_target": [False, True]},
    "HistGradientBoosting": {"learning_rate": [0.03, 0.05, 0.1], "max_iter": [100, 200, 400], "max_leaf_nodes": [15, 31, 63],
                             "l2_regularization": [0, 1, 10], "loss": ["squared_error", "absolute_error"],
                             "log_target": [False, True]},
}
N_ITER = {"Ridge": 8, "Random Forest": 8, "HistGradientBoosting": 20}

rows = []
for name, space in SPACES.items():
    for params in ParameterSampler(space, N_ITER[name], random_state=RS):
        scores, _ = fit_eval(make_model(name, params))
        rows.append({"model": name, "params": params, **scores})
search = pd.DataFrame(rows)
search["log_target"] = search["params"].map(lambda p: p["log_target"])
search["loss"] = search["params"].map(lambda p: p.get("loss"))

config.TABLES_DIR.mkdir(parents=True, exist_ok=True)
search.drop(columns=["log_target", "loss"]).round(2).to_csv(config.TABLES_DIR / "tuning_results.csv", index=False)
print(len(search), "configurations tested")
search.sort_values("MAE").drop(columns=["log_target", "loss"]).head(10).round(2)

36 configurations tested


,model,params,MAE,RMSE,RMSLE,mean error,fit seconds
34,HistGradientBoosting,"{'max_leaf_nodes': 31, 'max_iter': 400, 'loss': 'squared_error', 'log_target': True, '...",106.22,399.31,1.80,-81.38,19.08
27,HistGradientBoosting,"{'max_leaf_nodes': 63, 'max_iter': 100, 'loss': 'squared_error', 'log_target': True, '...",106.57,401.23,1.80,-80.92,9.85
16,HistGradientBoosting,"{'max_leaf_nodes': 15, 'max_iter': 400, 'loss': 'squared_error', 'log_target': True, '...",106.88,401.99,1.81,-83.61,12.99
18,HistGradientBoosting,"{'max_leaf_nodes': 63, 'max_iter': 100, 'loss': 'absolute_error', 'log_target': True, ...",107.48,397.95,1.89,-71.05,9.72
11,Random Forest,"{'n_estimators': 200, 'min_samples_split': 30, 'min_samples_leaf': 5, 'max_depth': 20,...",107.72,401.58,1.83,-78.83,37.10
26,HistGradientBoosting,"{'max_leaf_nodes': 63, 'max_iter': 400, 'loss': 'absolute_error', 'log_target': True, ...",107.94,392.21,1.88,-62.54,38.16
8,Random Forest,"{'n_estimators': 200, 'min_samples_split': 30, 'min_samples_leaf': 1, 'max_depth': 20,...",107.97,402.62,1.84,-77.55,39.55
13,Random Forest,"{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 20, 'max_depth': 10,...",108.22,400.92,1.88,-80.12,25.51
12,Random Forest,"{'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_depth': 10,...",108.22,402.64,1.85,-80.27,15.45
5,Ridge,"{'log_target': True, 'alpha': 10}",108.26,399.30,1.88,-80.76,0.35


The ten best configurations are close: MAE between 106.2 and 108.3 h, and they come from the three families (HistGradientBoosting, Random Forest and Ridge).

In [3]:
# effect of the log target (best MAE of each model), and of the loss for HistGradientBoosting
display(search.groupby(["model", "log_target"])["MAE"].min().unstack().round(1).rename(columns={False: "raw target", True: "log target"}))
hgb = search[search["model"] == "HistGradientBoosting"]
hgb.groupby(["loss", "log_target"])["MAE"].min().unstack().round(1).rename(columns={False: "raw target", True: "log target"})

log_target,raw target,log target
model,,
HistGradientBoosting,109.2,106.2
Random Forest,149.2,107.7
Ridge,162.4,108.3


log_target,raw target,log target
loss,,
absolute_error,109.2,107.5
squared_error,150.6,106.2


The log target gives the biggest improvement. MAE drops from 149.2 h to 107.7 h for Random Forest and from 162.4 h to 108.3 h for Ridge. HistGradientBoosting improves slightly, from 109.2 h to 106.2 h.

This confirms that long incidents affect squared error. Using the log target reduces this effect and improves MAE.

## 3. Best model of each family

In [4]:
best = search.loc[search.groupby("model")["MAE"].idxmin()].set_index("model")

# baseline: the median of the train target
base_pred = np.full(len(val), train[TARGET].median())
baseline = {**metrics.regression_metrics(val[TARGET], base_pred), "mean error": (base_pred - val[TARGET]).mean()}

comparison = pd.concat([pd.DataFrame([baseline], index=["Baseline (median)"]),
                        best[["MAE", "RMSE", "RMSLE", "mean error", "fit seconds"]]]).round(2)
comparison.to_csv(config.TABLES_DIR / "model_comparison_tuned.csv")
display(comparison)
best["params"]

,MAE,RMSE,RMSLE,mean error,fit seconds
Baseline (median),122.46,433.94,2.59,-90.40,NaN
HistGradientBoosting,106.22,399.31,1.80,-81.38,19.08
Random Forest,107.72,401.58,1.83,-78.83,37.10
Ridge,108.26,399.30,1.88,-80.76,0.35


model
HistGradientBoosting    {'max_leaf_nodes': 31, 'max_iter': 400, 'loss': 'squared_error', 'log_target': True, '...
Random Forest           {'n_estimators': 200, 'min_samples_split': 30, 'min_samples_leaf': 5, 'max_depth': 20,...
Ridge                                                                           {'log_target': True, 'alpha': 10}
Name: params, dtype: object

The three models perform better than the baseline on all metrics. The best MAE is 106.2 h against 122.5 h, but the overall improvement remains modest.

The models have very similar results. Ridge is almost as good as HistGradientBoosting and much faster (0.35 s vs 37.10 s). With only a few random configurations tested, it is difficult to say which model is truly better. I keep HistGradientBoosting because it has the lowest validation MAE, following the rule defined in notebook 05.

All models still underestimate the actual incident duration by around 80 hours on average.

## 4. Three checks on the best model

I test the best configuration by changing one thing at a time. All results use the same validation set.

- **Train from 28 March:** removes the first slow weeks (notebook 05).
- **Without `sys_created_by`:** removes a variable strongly linked to the target (notebook 04).
- **Without already `Resolved` incidents:** removes incidents whose status is not known at opening.

In [5]:
top = search.loc[search["MAE"].idxmin()]
name, params = top["model"], top["params"]
print("best configuration:", name, params)

variants = {
    "reference": {},
    "train from 28 March": {"train_df": train[train["opened_at"] >= "2016-03-28"]},
    "without sys_created_by": {"features": [f for f in FEATURES if f != "sys_created_by"]},
    "train without created Resolved": {"train_df": train[train["first_event_state"] != "Resolved"]},
}
rows, preds = [], {}
for label, kw in variants.items():
    scores, preds[label] = fit_eval(make_model(name, params), **kw)
    rows.append({"variant": label, "n train": len(kw.get("train_df", train)), **scores})
checks = pd.DataFrame(rows).set_index("variant").round(2)
checks.to_csv(config.TABLES_DIR / "sensitivity_checks.csv")
checks

best configuration: HistGradientBoosting {'max_leaf_nodes': 31, 'max_iter': 400, 'loss': 'squared_error', 'log_target': True, 'learning_rate': 0.03, 'l2_regularization': 0}


,n train,MAE,RMSE,RMSLE,mean error,fit seconds
variant,,,,,,
reference,14943,106.22,399.31,1.80,-81.38,15.89
train from 28 March,7353,106.41,395.19,1.81,-81.29,8.51
without sys_created_by,14943,107.52,400.31,1.92,-83.68,12.14
train without created Resolved,13629,106.67,399.75,1.84,-79.16,12.04


All variants use the same validation set. The differences are small, so changes below 1 h are not significant.

- **Train from 28 March:** MAE is 106.41 h instead of 106.22 h. Removing the first weeks brings no improvement, so I keep the full training set.
- **Without `sys_created_by`:** MAE increases to 107.52 h and RMSLE to 1.92. The variable helps slightly, so I keep it, but its influence is not fully understood.
- **Without already `Resolved` incidents:** MAE is 106.67 h. There is no improvement, so I keep these incidents.

## 5. Errors by duration

Where does the best model win or lose against the baseline? Incidents are grouped by their real duration on the validation set.

In [6]:
groups = pd.cut(val[TARGET], [-0.1, 1, 24, np.inf], labels=["under 1 h", "1 to 24 h", "over 24 h"])
model_pred = pd.Series(preds["reference"], index=val.index)
base_pred = pd.Series(base_pred, index=val.index)

pd.DataFrame({
    "n": groups.value_counts(),
    "MAE baseline": (val[TARGET] - base_pred).abs().groupby(groups, observed=True).mean(),
    "MAE model": (val[TARGET] - model_pred).abs().groupby(groups, observed=True).mean(),
    "mean error baseline": (base_pred - val[TARGET]).groupby(groups, observed=True).mean(),
    "mean error model": (model_pred - val[TARGET]).groupby(groups, observed=True).mean(),
}).round(1)

,n,MAE baseline,MAE model,mean error baseline,mean error model
resolution_hours,,,,,
under 1 h,2424,26.3,8.6,26.3,8.6
1 to 24 h,875,18.1,31.6,18.1,29.6
over 24 h,1682,315.3,285.8,-315.1,-268.7


The model performs well on short incidents but struggles with long ones.

- **Under 1 h:** MAE 8.6 h vs 26.3 h for the baseline.
- **1 to 24 h:** MAE 31.6 h vs 18.1 h.
- **Over 24 h:** MAE 285.8 h vs 315.3 h.

Long incidents account for about 90% of the total error. The model struggles to predict their duration, which remains the main limitation.

## Decisions

- **Final model:** HistGradientBoosting with log target, selected using validation MAE. Ridge gives similar results (108.3 h vs 106.2 h).
- **Features and training:** kept unchanged, including `sys_created_by` and resolved incidents.
- **Final training:** refitted on train + validation, then tested once on the test set (notebook 07).
- **Main limitation:** long incidents are underestimated and account for most of the error.